# Unit 12 / Chapter 12: Quantum Computer Vision

> **Main Learning Objective:** Encode tiny images into quantum states, build a mini Quantum Convolutional Neural Network (QCNN), and train it to classify 2x2 images by pattern.

| Section | Topic |
|---|---|
| 12.1 | How to load an image onto qubits (amplitude encoding, FRQI intuition) |
| 12.2 | Quantum convolutions and pooling |
| 12.3 | Building a full QCNN layer by layer |
| 12.4 | Training a QCNN on toy images |

---
## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100

# Tiny quantum simulator used across all units
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s
def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
def Rx(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)
def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])
def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1:
            bits[target] ^= 1
        y = 0
        for b in bits:
            y = (y<<1) | b
        op[y, x] = 1
    return op
def expZ(state, qubit, n):
    Zop = apply_1q(Z, qubit, n)
    return float(np.real(np.conj(state) @ Zop @ state))
print("Quantum simulator ready.")

---
## Course check-in

This logs that you started **Unit 12**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 12
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 12.1: How to Load an Image onto Qubits

A grayscale image is just a matrix of pixel intensities. A 2x2 image has 4 numbers. A 4x4 image has 16 numbers. In general an image with N pixels needs N numbers.

The naive approach is one qubit per pixel, but that blows up quickly. Instead quantum ML uses **amplitude encoding**: pack N pixel intensities into the amplitudes of a state on log2(N) qubits.

For a 2x2 image (4 pixels), we need only 2 qubits:

  |psi> = p00 |00> + p01 |01> + p10 |10> + p11 |11>

where each amplitude p is the (normalized) pixel intensity at that position. So a 4x4 image (16 pixels) needs 4 qubits, and an entire 256x256 image needs only 16 qubits. That is the exponential compression that makes quantum image processing interesting.

Two important encodings you will see in the literature:

* **Amplitude encoding**: cheap and compact, but requires state preparation that can be deep.
* **FRQI (Flexible Representation of Quantum Images)**: encodes intensity as a rotation on an ancilla qubit, keeping position on the address qubits. This decouples position from intensity, which is useful for image processing.

Below we amplitude-encode two toy 2x2 patterns and compare them via overlap.

In [ ]:
def amp_encode(image):
    v = np.array(image, dtype=float).flatten()
    if np.linalg.norm(v) == 0:
        raise ValueError("image is all zero")
    return v / np.linalg.norm(v)

# Two 2x2 patterns
diag = np.array([[1, 0], [0, 1]])
anti = np.array([[0, 1], [1, 0]])
horz = np.array([[1, 1], [0, 0]])

sd = amp_encode(diag); sa = amp_encode(anti); sh = amp_encode(horz)

print("Overlap diag vs anti-diag:", round(abs(np.vdot(sd, sa))**2, 3))
print("Overlap diag vs horizontal:", round(abs(np.vdot(sd, sh))**2, 3))

fig, axes = plt.subplots(1, 3, figsize=(6, 2))
for ax, img, name in zip(axes, [diag, anti, horz], ["diag", "anti-diag", "horiz"]):
    ax.imshow(img, cmap="gray_r"); ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()

### Activity 12.1

Encode the "vertical" pattern [[1,0],[1,0]] and compute its overlap with the horizontal pattern above. Confirm the two are orthogonal (overlap = 0).

In [ ]:
# TODO: amplitude-encode the vertical pattern and print its overlap with the
# horizontal pattern.
vert = None                # <-- np.array of shape (2,2)
sv   = None                # <-- amp_encode(vert)

if sv is not None:
    print("overlap vert vs horiz:", round(abs(np.vdot(sv, sh))**2, 3))
else:
    print("Fill in vert and sv, then re-run.")

<details><summary>Solution</summary>

```python
vert = np.array([[1, 0], [1, 0]])
sv   = amp_encode(vert)
```
The overlap is 0 because the two patterns share no non-zero pixel.
</details>

---
# Section 12.2: Quantum Convolutions and Pooling

A classical CNN has two ingredients:

* **Convolution** slides a small trainable kernel over the image.
* **Pooling** shrinks the image by averaging or taking the max of small regions.

A **QCNN** (Cong, Choi, Lukin 2019) does the same thing in quantum flavor:

* **Quantum convolution** is a parameterized 2-qubit gate applied to neighboring qubits. Since each pair of qubits addresses a local block of pixels in amplitude encoding, this is a local operation on the "image."
* **Quantum pooling** measures one qubit of each pair and uses the outcome to control a rotation on the surviving qubit. This trades off qubits for information, halving the qubit count each layer while keeping useful features.

The key advantage: a QCNN on n qubits requires only O(log n) trainable parameters, unlike a fully general parameterized quantum circuit that needs O(4^n). This is what makes QCNNs trainable and free of the barren-plateau problem for many tasks.

In [ ]:
def conv2q(theta, i, j, n):
    # Parameterized 2-qubit "convolution": Rz on each, CNOT, Ry on each
    U = apply_1q(Rz(theta[0]), i, n) @ apply_1q(Rz(theta[1]), j, n)
    U = apply_cnot(i, j, n) @ U
    U = apply_1q(Ry(theta[2]), i, n) @ apply_1q(Ry(theta[3]), j, n)
    return U

# Apply a convolution to two adjacent qubits of an amplitude-encoded image
theta = np.array([0.4, 0.8, 1.2, 1.6])
n = 2
state = sd.astype(complex)
state = conv2q(theta, 0, 1, n) @ state
print("post-convolution state:", np.round(state, 3))

---
# Section 12.3: Building a Full QCNN Layer by Layer

We stack a convolution layer, a pooling step, and a final measurement to produce a 1-number prediction. For 4 pixels (2 qubits) the QCNN is:

```
|image> -- conv2q(theta) -- pool(0->1) -- measure Z on qubit 1
```

Pooling here means: measure qubit 0. If it comes up 1, apply an Rx correction to qubit 1. We simulate this deterministically by computing <Z_1> given the post-convolution state.

In [ ]:
def qcnn_predict(theta, image):
    n = 2
    s = amp_encode(image).astype(complex)
    s = conv2q(theta, 0, 1, n) @ s
    # <Z_1> as the final prediction
    Z1 = apply_1q(Z, 1, n)
    return float(np.real(np.conj(s) @ Z1 @ s))

pred = qcnn_predict(theta, diag)
print("QCNN prediction on 'diag' with test theta:", round(pred, 3))

---
# Section 12.4: Training a QCNN on Toy Images

Task: distinguish "diagonal-like" images from "horizontal-like" ones. Diagonals get label +1, horizontals get label -1. We grid-search theta to minimize squared error.

In [ ]:
train_images = [
    (diag,   +1),
    (anti,   +1),   # both diagonals
    (horz,   -1),
    (np.array([[0,0],[1,1]]), -1),
]

def train_loss(theta):
    err = 0.0
    for img, y in train_images:
        p = qcnn_predict(theta, img)
        err += (p - y)**2
    return err / len(train_images)

best_L = 1e9; best = None
for a in np.linspace(0, 2*np.pi, 8):
    for b in np.linspace(0, 2*np.pi, 8):
        for c in np.linspace(0, 2*np.pi, 8):
            for d in np.linspace(0, 2*np.pi, 8):
                L = train_loss([a,b,c,d])
                if L < best_L:
                    best_L = L; best = [a,b,c,d]
print("best loss:", round(best_L, 3))
for img, y in train_images:
    p = qcnn_predict(best, img)
    print(f"  label={y:+d}, pred={p:+.2f}, correct={'yes' if np.sign(p)==y else 'no'}")

### Activity 12.2

Create a brand-new diagonal-looking image (say [[0.9, 0.1], [0.05, 0.95]]) and predict its class using the trained theta. You should see a positive prediction.

In [ ]:
# TODO: build a new diagonal image and print its QCNN prediction.
new_img = None       # <-- np.array of shape (2,2)

if new_img is not None:
    p = qcnn_predict(best, new_img)
    print(f"prediction on new diagonal-like image: {p:+.3f}  ({'diag' if p > 0 else 'horiz'})")
else:
    print("Fill in new_img, then re-run.")

<details><summary>Solution</summary>

```python
new_img = np.array([[0.9, 0.1], [0.05, 0.95]])
```
Prediction should be positive (roughly +0.9) because the pattern is close to the training diagonals.
</details>

### Activity 12.3

Explain in one sentence why a QCNN can process a 256x256 image using only 16 qubits, and what the trade-off is.

In [ ]:
answer_12_3 = """YOUR ANSWER HERE, one or two sentences."""
print(answer_12_3)

<details><summary>Sample answer</summary>

Amplitude encoding stores N pixels in log2(N) qubits, so 65,536 pixels fit in 16 qubits. The trade-off is that the amplitudes are only accessible through measurement statistics, so reading a full image back out requires many circuit repetitions.
</details>

---
## Section summary

* Amplitude encoding packs N pixels into log2(N) qubits, giving exponential compression.
* Quantum convolutions are small parameterized 2-qubit gates that act locally.
* Quantum pooling halves the qubit count each layer, giving QCNNs O(log n) parameters and helping them avoid barren plateaus.
* Even a 2-qubit QCNN can classify 4-pixel patterns after training.

---
## End-of-Unit Quiz (10 multiple choice)

**Q1.** How many qubits are needed to amplitude-encode a 4x4 grayscale image?

A. 2
B. 4
C. 8
D. 16

**Q2.** Amplitude encoding stores pixel intensities as:

A. Basis state indices
B. Amplitudes of the quantum state
C. Phases only
D. Frequencies

**Q3.** FRQI differs from plain amplitude encoding in that:

A. It uses no qubits
B. It encodes intensity as a rotation on an ancilla, keeping position on address qubits
C. It only works for binary images
D. It requires more qubits than pixels

**Q4.** A QCNN quantum convolution acts on:

A. All qubits at once
B. Neighboring pairs of qubits with a small parameterized gate
C. Only classical bits
D. Only the ancilla register

**Q5.** QCNN pooling reduces:

A. The number of qubits by half each layer
B. The kernel size
C. The dataset size
D. The learning rate

**Q6.** How many trainable parameters does a QCNN on n qubits use, asymptotically?

A. O(2^n)
B. O(4^n)
C. O(log n)
D. O(1)

**Q7.** Why does the QCNN structure help avoid barren plateaus?

A. It uses more qubits
B. Its structure is local and hierarchical rather than fully connected
C. It skips the training step
D. It uses only classical layers

**Q8.** In our 2-qubit QCNN, the prediction was computed as:

A. The largest amplitude
B. <Z> on the surviving qubit after pooling
C. The sum of all amplitudes
D. A random number

**Q9.** The overlap between two orthogonal 2x2 patterns like horizontal and vertical is:

A. 0
B. 0.5
C. 1
D. Undefined

**Q10.** A key trade-off of amplitude encoding is that:

A. The state can only be read out through measurement statistics, requiring many shots
B. It cannot represent grayscale
C. It is limited to 2 qubits
D. It requires classical convolution

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 12!")